# Pretraining and the gate - bounding what the surviving skill actually is

Two knobs that every arm in every grid so far has held fixed, and that each bound a different
alternative explanation for the results. Neither requires touching the library or any existing
notebook; both are pure config arms on the otherwise unmodified full model.

| arm | change | why |
|---|---|---|
| `baseline` | none (imported) | reference |
| `no_pretrain` | skip the 1-epoch self-supervised pass entirely | separates "the encoder is useless" from "the encoder is useful only through pretraining" |
| `gate_avg` | `agg_measure='avg'` - flat 0.5 gate | removes the degree-dependent gate, which is real signal independent of both encoder and signature |
| `gate_100proto` | `agg_measure='100proto'` - prototype only | the complement of `proto_off`: how good is the pooled prototype *alone*? |
| `gate_heuristic` | `agg_measure='heuristics-0.8'` - flat 0.2 gate | a second flat-gate point, to show the result is not specific to 0.5 |

## Why pretraining matters for the argument

Every reported arm runs one epoch of self-supervised pretraining over **all** relation types
(~6,578 steps at the full KG) before 1,000 epochs of drug-disease fine-tuning. That is the only stage at
which most of the knowledge graph is ever a training signal: fine-tuning only ever back-propagates
through drug-disease edges.

This creates an ambiguity in the headline encoder result. `rgcn -> light_gnn` costs -0.227 indication,
which was attributed to the R-GCN's 1.01M relation-specific parameters. But those parameters are also
the only thing that the one pretraining epoch can write into. If `no_pretrain` collapses, then
"the encoder matters" is really "pretraining matters, and the encoder is where it is stored" - a
materially different claim, and one that *strengthens* the knowledge-graph story rather than weakening
it. If `no_pretrain` barely moves, pretraining can be removed from the causal story altogether.

`no_pretrain` is also the **cheapest run in the whole study**: dropping the pretraining pass removes
~20 minutes of the ~25-minute run.

## Why the gate matters for the argument

With `agg_measure='rarity'` the mixing coefficient is

    c_i = exp(-0.7 * deg_i)  (via `exponential`, model.py:259)

a function of the disease's **drug-disease degree in the training graph**. On zero-shot test diseases
that degree is 0, so `c_i ~ 0.9` and the embedding is ~90% prototype. Two consequences:

1. The gate itself carries information - degree is a real feature - so part of what looks like
   "metric learning works" could be "the model knows which diseases are held out".
2. Because `c_i` is near-saturated in the zero-shot regime, a flat 0.5 gate should *reduce* the
   prototype's influence, and `100proto` should *increase* it slightly. Measuring both brackets the
   gate's contribution without needing to model it.

`gate_avg` is the control named in `docs/ABLATION_FINDINGS.md` section 11. `gate_100proto` is its
natural complement and the one that pairs with `proto_off` from the signature grid: `proto_off` removes
the prototype, `100proto` keeps nothing but.

---
## 1 - Environment

In [ ]:
import os, sys, json, time, gc, random, shutil, inspect, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')


def _find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'pyg_implementation' / 'txgnn').is_dir():
            return cand
    raise RuntimeError('Could not find pyg_implementation/txgnn above ' + str(p))


REPO_ROOT = _find_repo_root()
PYG_ROOT  = REPO_ROOT / 'pyg_implementation'
sys.path.insert(0, str(PYG_ROOT))

import torch
import torch.nn as nn

import txgnn
from txgnn import TxData, TxGNN
from txgnn.utils import evaluate_fb, create_pyg_graph, get_n_params

assert 'pyg_implementation' in txgnn.__file__, (
    'Wrong txgnn package on sys.path (%s). Restart the kernel.' % txgnn.__file__)

_MI_PARAMS = set(inspect.signature(TxGNN.model_initialize).parameters)
assert 'use_decision_network' not in _MI_PARAMS, (
    'model_initialize accepts use_decision_network again - revisit this notebook.')
assert 'agg_measure' in _MI_PARAMS, 'model_initialize no longer takes agg_measure'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print('repo  :', REPO_ROOT)
print('txgnn :', txgnn.__file__)
print('torch :', torch.__version__, '| device:', DEVICE,
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor())

---
## 2 - Configuration

`MODEL_CFG` and the schedule are identical to every other grid, so `baseline` imports free from
`ablation_encoder_swap/rgcn` and all deltas are directly comparable to the published ones.

In [ ]:
SPLIT   = 'complex_disease'
SEEDS   = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 31, 41, 42, 63, 70, 77, 86, 89, 95, 98]
PROFILE = 'full'

MODEL_CFG = dict(
    n_hid       = 100,
    n_inp       = 100,
    n_out       = 100,
    proto       = True,
    proto_num   = 3,
    attention   = False,
    sim_measure = 'all_nodes_profile',
    agg_measure = 'rarity',              # the arms below override exactly this
)

PRETRAIN_CFG = dict(n_epoch=1,    learning_rate=1e-3, batch_size=1024, train_print_per_n=500)
FINETUNE_CFG = dict(n_epoch=1000, learning_rate=5e-4, train_print_per_n=100, valid_per_n=20)

DD_REL     = ['contraindication', 'indication', 'off-label use']
DD_REV_REL = ['rev_' + r for r in DD_REL]
DD_ETYPES  = [('drug', r, 'disease') for r in DD_REL] + [('disease', r, 'drug') for r in DD_REV_REL]

SIG_ETYPES_DEFAULT = ['disease_disease', 'rev_disease_protein']

DATA_FOLDER = REPO_ROOT / 'data'
ABL_ROOT    = REPO_ROOT / 'results' / 'ablation_pretrain_and_gate'
SWAP_ROOT   = REPO_ROOT / 'results' / 'ablation_encoder_swap'   # read-only: baseline import
ABL_ROOT.mkdir(parents=True, exist_ok=True)

REF_ARM = 'baseline'

# --- switches ------------------------------------------------------------------------------------
RUN_GRID             = True
SAVE_MODELS          = False
IMPORT_REF           = True
INCLUDE_100PROTO     = False   # +1 arm, ~8 GPU-h: prototype-only gate
INCLUDE_HEURISTIC    = False   # +1 arm, ~8 GPU-h: second flat-gate point (0.2)
STOP_AFTER_N_RUNS    = None

print('split  :', SPLIT)
print('seeds  : %d -> %s' % (len(SEEDS), SEEDS))
print('results:', ABL_ROOT.relative_to(REPO_ROOT))

---
## 3 - The arms

`overrides` go into `model_initialize`; `pretrain` is `None` for the arm that skips the self-supervised
pass. Nothing else differs between arms - same graph, same relations, same metric learning, same
fine-tuning schedule.

In [ ]:
ARMS = [
    {'arm': 'baseline',  'overrides': {}, 'pretrain': PRETRAIN_CFG,
     'note': 'reference: rarity gate + 1 pretrain epoch (imported)'},
    {'arm': 'no_pretrain', 'overrides': {}, 'pretrain': None,
     'note': 'no self-supervised pass at all; finetune only'},
    {'arm': 'gate_avg',  'overrides': {'agg_measure': 'avg'}, 'pretrain': PRETRAIN_CFG,
     'note': 'flat 0.5 gate: removes the degree dependence'},
]
if INCLUDE_100PROTO:
    ARMS.append({'arm': 'gate_100proto', 'overrides': {'agg_measure': '100proto'},
                 'pretrain': PRETRAIN_CFG,
                 'note': 'prototype only; complement of proto_off'})
if INCLUDE_HEURISTIC:
    ARMS.append({'arm': 'gate_heuristic', 'overrides': {'agg_measure': 'heuristics-0.8'},
                 'pretrain': PRETRAIN_CFG,
                 'note': 'flat 0.2 gate; second flat point'})

ARMS_BY_NAME = {a['arm']: a for a in ARMS}
assert len(ARMS_BY_NAME) == len(ARMS), 'duplicate arm name'
assert REF_ARM in ARMS_BY_NAME, 'reference arm missing'

VALID_AGG = {'learn', 'heuristics-0.8', 'avg', 'rarity', '100proto'}   # model.py:248-290
for a in ARMS:
    agg = dict(MODEL_CFG, **a['overrides'])['agg_measure']
    assert agg in VALID_AGG, 'unsupported agg_measure %r' % agg

_new = [a for a in ARMS if a['arm'] != REF_ARM]
print('%d arms x %d seeds | %d need GPU time (baseline imports free)\n'
      % (len(ARMS), len(SEEDS), len(_new) * len(SEEDS)))
for a in ARMS:
    cfg = dict(MODEL_CFG, **a['overrides'])
    print('  %-16s agg_measure=%-14s pretrain=%s'
          % (a['arm'], cfg['agg_measure'],
             'n_epoch=%d' % a['pretrain']['n_epoch'] if a['pretrain'] else 'SKIPPED'))
    print('  %-16s %s' % ('', a['note']))

---
## 4 - Mechanics

`no_pretrain` does **not** call `m.pretrain(...)` at all, rather than passing `n_epoch=0`. Both would
work - `TxGNN.pretrain` loops over `range(n_epoch)` - but `n_epoch=0` would still spend several minutes
building the flat edge list and the minibatch dataloader for nothing. Skipping is also unambiguous in
the metadata, which records `pretrain: null`.

`finetune` is self-contained: it builds its own `AdamW` over `self.model.parameters()` and re-initialises
the decoder (`torch.nn.init.xavier_uniform(self.model.w_rels)`, `TxGNN.py:231`), so it does not depend
on pretraining having run. `best_model` is set by `model_initialize` and then replaced by `finetune` at
each validation improvement, so it is always defined; `run_one` asserts it anyway.

In [ ]:
class RunData:
    '''Minimal TxData stand-in. Graph rebuilt per run; nothing filtered in this notebook.'''

    def __init__(self, base, data_folder, split):
        self.df       = base['df']
        self.df_train = base['df_train']
        self.df_valid = base['df_valid']
        self.df_test  = base['df_test']
        self.G = create_pyg_graph(self.df_train, self.df)
        self.data_folder = str(data_folder)
        self.split = split
        self.disease_eval_idx = None
        self.no_kg = False


_SEED_CACHE = {}


def get_seed_data(seed):
    if seed in _SEED_CACHE:
        return _SEED_CACHE[seed]
    _SEED_CACHE.clear()
    gc.collect()
    print('[data] loading split seed %d ...' % seed)
    t0 = time.time()
    set_all_seeds(seed)
    d = TxData(data_folder_path=str(DATA_FOLDER))
    d.prepare_split(split=SPLIT, seed=seed, no_kg=False)
    base = {'df': d.df, 'df_train': d.df_train, 'df_valid': d.df_valid, 'df_test': d.df_test}
    del d
    gc.collect()
    _SEED_CACHE[seed] = base
    print('[data] seed %d ready in %.1f s | train edges %d'
          % (seed, time.time() - t0, len(base['df_train'])))
    return base


def run_dir(arm, seed):
    return ABL_ROOT / ('%s_seed%d' % (arm, seed))


def is_done(arm, seed):
    d = run_dir(arm, seed)
    return (d / 'edge_level_test_metrics.csv').exists() and (d / 'run_meta.json').exists()


def encoder_params(model):
    return (sum(p.numel() for p in model.layer1.parameters())
            + sum(p.numel() for p in model.layer2.parameters()))


def gate_probe(model):
    '''Report the mixing coefficient the model actually used, where it is recoverable.

    For the flat gates the coefficient is a constant written into model.py, so it is reported from
    agg_measure directly. For `rarity` it is per-disease and depends on degree, so only the configured
    mode is recorded. This exists so the metadata states the gate unambiguously.
    '''
    agg = getattr(model.pred, 'agg_measure', None)
    const = {'avg': 0.5, 'heuristics-0.8': 0.2, '100proto': 1.0}.get(agg)
    return {'agg_measure': agg, 'gate_constant': const,
            'gate_is_degree_dependent': agg == 'rarity'}

In [ ]:
# --- import the baseline --------------------------------------------------------------------------
MODEL_KEYS_CHECKED = sorted(MODEL_CFG)


def import_matching_run(src, dst, expect, extra=None):
    sm, se = src / 'run_meta.json', src / 'edge_level_test_metrics.csv'
    if not (sm.exists() and se.exists()):
        return False, 'no run at %s' % src.name
    meta = json.loads(sm.read_text())
    for k, want in expect.items():
        got = meta.get(k, '<missing>')
        if isinstance(want, dict):
            if not isinstance(got, dict):
                return False, '%s is not a dict' % k
            bad = {kk: (got.get(kk, '<missing>'), vv) for kk, vv in want.items()
                   if got.get(kk, '<missing>') != vv}
            if bad:
                return False, '%s mismatch %s' % (k, bad)
        elif got != want:
            return False, '%s mismatch: %r != %r' % (k, got, want)

    dst.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(se, dst / 'edge_level_test_metrics.csv')
    meta.update(extra or {})
    meta['imported_from'] = str(src.relative_to(REPO_ROOT))
    meta['run_key'] = dst.name
    meta['arm'] = dst.name.rsplit('_seed', 1)[0]
    with open(dst / 'run_meta.json', 'w') as f:
        json.dump(meta, f, indent=2, default=str)
    return True, 'imported from %s' % src.relative_to(REPO_ROOT)


def try_import_ref(seed):
    if not IMPORT_REF:
        return False, 'disabled'
    return import_matching_run(
        SWAP_ROOT / ('rgcn_seed%d' % seed), run_dir(REF_ARM, seed),
        expect={'split': SPLIT, 'seed': seed, 'dropped_edges_total': 0, 'proto': True,
                'encoder': 'rgcn',
                'signature_disease_etypes': SIG_ETYPES_DEFAULT,
                'model_config': {k: MODEL_CFG[k] for k in MODEL_KEYS_CHECKED},
                'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG},
        extra={'experiment': 'pretrain_and_gate', 'pretrain_skipped': False,
               'gate': {'agg_measure': 'rarity', 'gate_constant': None,
                        'gate_is_degree_dependent': True}})


if IMPORT_REF:
    n_ok = 0
    for s in SEEDS:
        if is_done(REF_ARM, s):
            n_ok += 1
            continue
        ok, why = try_import_ref(s)
        n_ok += bool(ok)
        if not ok:
            print('[import] %s_seed%d: will train (%s)' % (REF_ARM, s, why))
    print('[import] %s %d/%d available' % (REF_ARM, n_ok, len(SEEDS)))

# --- plan -----------------------------------------------------------------------------------------
_pre, _fin = [], []
for _p in sorted(SWAP_ROOT.glob('rgcn_seed*/run_meta.json')):
    _r = json.loads(_p.read_text())
    if _r.get('pretrain_seconds') and _r.get('finetune_seconds'):
        _pre.append(_r['pretrain_seconds'])
        _fin.append(_r['finetune_seconds'])
PRE_MIN = (float(np.mean(_pre)) / 60) if _pre else 19.7
FIN_MIN = (float(np.mean(_fin)) / 60) if _fin else 4.9
print('\n[timing] baseline averages %.1f min pretrain + %.1f min finetune = %.1f min'
      % (PRE_MIN, FIN_MIN, PRE_MIN + FIN_MIN))

rows, total_min = [], 0.0
for a in ARMS:
    done = [s for s in SEEDS if is_done(a['arm'], s)]
    todo = len(SEEDS) - len(done)
    per = FIN_MIN if a['pretrain'] is None else PRE_MIN + FIN_MIN
    est = 0.0 if a['arm'] == REF_ARM else todo * per
    total_min += est
    cfg = dict(MODEL_CFG, **a['overrides'])
    rows.append({'arm': a['arm'], 'agg_measure': cfg['agg_measure'],
                 'pretrain': 'skipped' if a['pretrain'] is None else '1 epoch',
                 'min/run': round(per, 1), 'done': len(done), 'todo': todo,
                 'est GPU-h': round(est / 60, 2)})
display(pd.DataFrame(rows).style.hide(axis='index'))

RUN_KEYS = [(a['arm'], s) for s in SEEDS for a in ARMS]
pending = [k for k in RUN_KEYS if not is_done(*k)]
print('\n%d run_keys, %d done, %d pending | estimated %.1f GPU-hours remaining'
      % (len(RUN_KEYS), len(RUN_KEYS) - len(pending), len(pending), total_min / 60))
for arm, seed in RUN_KEYS:
    print('  [%s] %s_seed%d' % ('x' if is_done(arm, seed) else ' ', arm, seed))

---
## 5 - The grid

In [ ]:
def run_one(arm_name, seed, verbose=True):
    arm  = ARMS_BY_NAME[arm_name]
    rdir = run_dir(arm_name, seed)
    el_csv, meta_json = rdir / 'edge_level_test_metrics.csv', rdir / 'run_meta.json'

    if el_csv.exists() and meta_json.exists():
        if verbose:
            print('[skip] %s_seed%d -> cached' % (arm_name, seed))
        return pd.read_csv(el_csv)

    if arm_name == REF_ARM:
        ok, why = try_import_ref(seed)
        if ok:
            print('[skip] %s_seed%d -> %s' % (arm_name, seed, why))
            return pd.read_csv(el_csv)

    cfg = dict(MODEL_CFG, **arm['overrides'])

    print('\n' + '#' * 78)
    print('# %s   seed %d   agg_measure=%s   pretrain=%s'
          % (arm_name, seed, cfg['agg_measure'],
             'n_epoch=%d' % arm['pretrain']['n_epoch'] if arm['pretrain'] else 'SKIPPED'))
    print('#' * 78)

    base = get_seed_data(seed)
    set_all_seeds(seed)
    d = RunData(base, DATA_FOLDER, SPLIT)
    rel_names = {et[1] for et in d.G.edge_types}
    assert all(r in rel_names for r in SIG_ETYPES_DEFAULT), 'KG is not intact'
    etypes_after = sorted('%s|%s|%s' % et for et in d.G.edge_types)
    print('[graph] train edges %d | edge types %d (KG intact in every arm)'
          % (len(d.df_train), len(etypes_after)))

    set_all_seeds(seed)
    m = TxGNN(data=d, weight_bias_track=False, exp_name='%s_seed%d' % (arm_name, seed), device=DEVICE)
    m.model_initialize(**cfg)

    # the config the model is actually holding, not the one we asked for
    assert m.model.pred.agg_measure == cfg['agg_measure'], \
        'gate is %r, expected %r' % (m.model.pred.agg_measure, cfg['agg_measure'])
    gate = gate_probe(m.model)
    print('[model] encoder params %s | total %s | gate %s'
          % (format(encoder_params(m.model), ','), format(get_n_params(m.model), ','), gate))

    pre_s = 0.0
    if arm['pretrain'] is not None:
        t0 = time.time()
        set_all_seeds(seed)
        m.pretrain(**arm['pretrain'])
        pre_s = time.time() - t0
    else:
        print('[pretrain] SKIPPED - finetune builds its own optimiser and re-inits w_rels')

    t0 = time.time()
    set_all_seeds(seed)
    m.finetune(**FINETUNE_CFG)
    fin_s = time.time() - t0

    assert m.best_model is not None, 'finetune left best_model unset'

    (auroc_rel, auprc_rel, micro_auroc, micro_auprc, macro_auroc, macro_auprc), test_loss = \
        evaluate_fb(m.best_model, m.g_test_pos, m.g_test_neg, m.G, m.dd_etypes, DEVICE, mode='test')

    el = pd.DataFrame(
        [('%s --%s--> %s' % et, auroc_rel.get(et, np.nan), auprc_rel.get(et, np.nan))
         for et in DD_ETYPES],
        columns=['relation', 'AUROC', 'AUPRC'])
    el.loc[len(el)] = ['MICRO (drug-disease)', micro_auroc, micro_auprc]
    el.loc[len(el)] = ['MACRO (drug-disease)',
                       float(np.nanmean([auroc_rel.get(et, np.nan) for et in DD_ETYPES])),
                       float(np.nanmean([auprc_rel.get(et, np.nan) for et in DD_ETYPES]))]
    el.loc[len(el)] = ['MACRO (all relations)', macro_auroc, macro_auprc]

    rdir.mkdir(parents=True, exist_ok=True)
    el.to_csv(el_csv, index=False)

    meta = {
        'run_key': '%s_seed%d' % (arm_name, seed),
        'arm': arm_name, 'seed': seed, 'split': SPLIT, 'profile': PROFILE,
        'experiment': 'pretrain_and_gate',
        'note': arm['note'],
        'gate': gate,
        'pretrain_skipped': arm['pretrain'] is None,
        'encoder': 'rgcn',
        'encoder_params': encoder_params(m.model),
        'n_params': get_n_params(m.model),
        'proto': cfg['proto'],
        'signature_disease_etypes': SIG_ETYPES_DEFAULT,
        'dropped_edges_total': 0,
        'train_edges': len(d.df_train),
        'n_edge_types_after': len(etypes_after),
        'edge_types_after': etypes_after,
        'model_config': m.config,
        'pretrain': arm['pretrain'], 'finetune': FINETUNE_CFG,
        'pretrain_seconds': pre_s, 'finetune_seconds': fin_s,
        'test_bce_loss': float(test_loss),
        'torch': torch.__version__,
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'finished_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    }
    with open(meta_json, 'w') as f:
        json.dump(meta, f, indent=2, default=str)

    if SAVE_MODELS:
        mdir = REPO_ROOT / 'saved_models' / 'ablation_pretrain_and_gate' / meta['run_key']
        mdir.mkdir(parents=True, exist_ok=True)
        m.save_model(str(mdir))

    ind = auprc_rel.get(('drug', 'indication', 'disease'), np.nan)
    con = auprc_rel.get(('drug', 'contraindication', 'disease'), np.nan)
    print('[done] %s_seed%d | %.1f min | indication %.4f | contraindication %.4f -> %s'
          % (arm_name, seed, (pre_s + fin_s) / 60, ind, con, rdir.relative_to(REPO_ROOT)))

    del m, d, auroc_rel, auprc_rel
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return el

In [ ]:
if RUN_GRID:
    todo = [(a['arm'], s) for s in SEEDS for a in ARMS if not is_done(a['arm'], s)]
    if STOP_AFTER_N_RUNS:
        todo = todo[:STOP_AFTER_N_RUNS]
    print('%d run(s) pending of %d total.' % (len(todo), len(ARMS) * len(SEEDS)))
    grid_t0 = time.time()
    for i, (arm_name, seed) in enumerate(todo, 1):
        print('\n>>> %d/%d  %s_seed%d  (elapsed %.1f h)'
              % (i, len(todo), arm_name, seed, (time.time() - grid_t0) / 3600))
        run_one(arm_name, seed)
    print('\nGrid finished %d run(s) in %.1f h.' % (len(todo), (time.time() - grid_t0) / 3600))
else:
    print('RUN_GRID is False - no training. Set it in section 2.')

---
## 6 - Aggregate

In [ ]:
def _load_all(root):
    recs = []
    for meta_p in sorted(Path(root).glob('*/run_meta.json')):
        el_p = meta_p.parent / 'edge_level_test_metrics.csv'
        if not el_p.exists():
            continue
        meta = json.loads(meta_p.read_text())
        el = pd.read_csv(el_p).set_index('relation')
        gate = meta.get('gate') or {}
        for rel, row in el.iterrows():
            recs.append({'arm': meta['arm'], 'seed': meta['seed'], 'relation': rel,
                         'AUROC': row['AUROC'], 'AUPRC': row['AUPRC'],
                         'agg_measure': gate.get('agg_measure'),
                         'pretrain_skipped': meta.get('pretrain_skipped'),
                         'pretrain_seconds': meta.get('pretrain_seconds'),
                         'imported': 'imported_from' in meta})
    return pd.DataFrame(recs)


def _ci95(x):
    x = np.asarray(x, dtype=float)
    return 1.96 * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan


ROW = {'indication': 'drug --indication--> disease',
       'contraindication': 'drug --contraindication--> disease',
       'micro': 'MICRO (drug-disease)'}

long = _load_all(ABL_ROOT)
summary = wide = ref = None

if long.empty:
    print('No runs on disk yet - run section 5 first.')
else:
    wide = (long[long.relation.isin(ROW.values())]
            .pivot_table(index=['arm', 'seed'], columns='relation', values='AUPRC')
            .rename(columns={v: k for k, v in ROW.items()}))
    info = long.groupby('arm').agg(agg_measure=('agg_measure', 'first'),
                                   pretrain_skipped=('pretrain_skipped', 'first'))
    present = set(wide.index.get_level_values('arm'))
    ref = wide.xs(REF_ARM, level='arm') if REF_ARM in present else None
    if ref is None:
        print('[warn] no %s runs - deltas will be NaN.' % REF_ARM)

    out = []
    for arm, g in wide.groupby(level='arm'):
        g = g.droplevel('arm')
        rec = {'arm': arm, 'n_seeds': len(g),
               'agg_measure': info.loc[arm, 'agg_measure'],
               'pretrain_skipped': info.loc[arm, 'pretrain_skipped']}
        for key in ROW:
            rec['%s_AUPRC' % key] = g[key].mean()
            rec['%s_CI' % key] = _ci95(g[key])
            if arm == REF_ARM:
                rec['%s_delta' % key] = 0.0
                rec['%s_delta_CI' % key] = np.nan
            elif ref is not None:
                paired = (g[key] - ref[key]).dropna()
                rec['%s_delta' % key] = paired.mean() if len(paired) else np.nan
                rec['%s_delta_CI' % key] = _ci95(paired)
                rec['%s_hurt' % key] = int((paired < 0).sum())
                rec['%s_n_paired' % key] = len(paired)
        out.append(rec)

    order = [a['arm'] for a in ARMS]
    summary = (pd.DataFrame(out)
                 .assign(_o=lambda d: d.arm.map({a: i for i, a in enumerate(order)}))
                 .sort_values('_o').drop(columns='_o').reset_index(drop=True))
    summary.to_csv(ABL_ROOT / 'pretrain_and_gate_summary.csv', index=False)

    cols = [c for c in ['arm', 'n_seeds', 'agg_measure', 'pretrain_skipped',
                        'indication_AUPRC', 'indication_CI', 'indication_delta',
                        'indication_delta_CI', 'indication_hurt',
                        'contraindication_AUPRC', 'contraindication_delta',
                        'contraindication_delta_CI', 'contraindication_hurt',
                        'micro_AUPRC', 'micro_delta'] if c in summary.columns]
    print('%d arms, %d runs -> %s\n'
          % (len(summary), len(wide),
             (ABL_ROOT / 'pretrain_and_gate_summary.csv').relative_to(REPO_ROOT)))
    display(summary[cols].style
            .format('{:.4f}', subset=[c for c in cols if c.endswith(('AUPRC', 'delta', 'CI'))],
                    na_rep='-')
            .hide(axis='index'))

### 6.1 - Reading both answers against the published numbers

In [ ]:
FLOOR = 0.500
PUBLISHED = {            # for context, from the other grids (n=20 unless noted)
    'rgcn -> light_gnn':      (-0.2266, -0.1095),
    'rgcn -> mlp':            (-0.3033, -0.1139),
    'proto_off (n=10)':       (-0.1737, -0.1482),
    'keep 9/50 edge types':   (-0.0658, -0.0506),
    'drop disease_protein':   (-0.1078, -0.1258),
}

if summary is None or ref is None:
    print('Need finished %s runs plus at least one other arm.' % REF_ARM)
else:
    s = summary.set_index('arm')
    head = s.loc[REF_ARM, 'indication_AUPRC']
    rows = []
    for arm in [a['arm'] for a in ARMS if a['arm'] in s.index and a['arm'] != REF_ARM]:
        rows.append({'change': arm,
                     'indication delta': s.loc[arm, 'indication_delta'],
                     '% of headroom': 100 * -s.loc[arm, 'indication_delta'] / (head - FLOOR),
                     'contraind. delta': s.loc[arm, 'contraindication_delta'],
                     'n': int(s.loc[arm, 'n_seeds'])})
    for k, (i, c) in PUBLISHED.items():
        rows.append({'change': '(published) ' + k, 'indication delta': i,
                     '% of headroom': 100 * -i / (head - FLOOR), 'contraind. delta': c, 'n': 20})
    tbl = pd.DataFrame(rows).sort_values('indication delta')
    display(tbl.style.format({'indication delta': '{:+.4f}', 'contraind. delta': '{:+.4f}',
                              '% of headroom': '{:.1f}'}, na_rep='-').hide(axis='index'))

    print('\n' + '=' * 78)
    if 'no_pretrain' in s.index:
        d = s.loc['no_pretrain', 'indication_delta']
        print('PRETRAINING: removing it costs %+.4f indication (%.1f%% of headroom).' % (
            d, 100 * -d / (head - FLOOR)))
        if -d > 0.15:
            print('  -> Large. The encoder result is substantially a PRETRAINING result: the 1.01M')
            print('     relation-specific parameters matter because one self-supervised epoch writes')
            print('     the whole KG into them. Re-word the encoder claim in the paper draft.')
        elif -d < 0.03:
            print('  -> Negligible. Pretraining can be dropped from the causal story entirely, and')
            print('     the encoder result is about capacity at fine-tuning time.')
        else:
            print('  -> Moderate. Report it alongside the encoder result; neither explains the other.')
    if 'gate_avg' in s.index:
        d = s.loc['gate_avg', 'indication_delta']
        print('\nGATE: flattening rarity -> 0.5 costs %+.4f indication (%.1f%% of headroom).' % (
            d, 100 * -d / (head - FLOOR)))
        print('  Compare with proto_off (-0.1737): the gate accounts for %.0f%% of what removing'
              % (100 * -d / 0.1737))
        print('  metric learning altogether costs. The remainder is the pooled prototype itself.')
    if 'gate_100proto' in s.index:
        print('\n  100proto (prototype only): %+.4f indication - the upper end of the gate axis.'
              % s.loc['gate_100proto', 'indication_delta'])

In [ ]:
# Per-seed paired deltas.
if summary is not None and ref is not None:
    for key in ['indication', 'contraindication']:
        arms = [a for a in summary.arm if a != REF_ARM
                and a in wide.index.get_level_values('arm')]
        if not arms:
            continue
        tbl = pd.DataFrame({a: (wide.xs(a, level='arm')[key] - ref[key]) for a in arms}).T
        seeds_present = list(tbl.columns)
        tbl.columns = ['seed %d' % x for x in seeds_present]
        tbl['mean'] = tbl.mean(axis=1)
        tbl['hurt'] = (tbl[['seed %d' % x for x in seeds_present]] < 0).sum(axis=1)
        print('\npaired delta AUPRC vs %s - %s  (negative = the change hurt)' % (REF_ARM, key))
        display(tbl.style.format('{:.4f}', subset=[c for c in tbl.columns if c != 'hurt'],
                                 na_rep='-'))

---
## 7 - What this settles

These two arms close the last two "smaller gaps" listed in `docs/ABLATION_FINDINGS.md` section 11.
After they run, the remaining open items in the whole study are the protocol question
(`ablation_full_ranking.ipynb`) and the frozen-feature question
(`ablation_learnable_embeddings.ipynb`) - not the mechanism questions.

Note one asymmetry worth stating in the write-up: `no_pretrain` is the only arm in the entire study
that makes the model **cheaper** rather than smaller. If it costs little, that is a practical finding
about TxGNN independent of any claim about attribution - 80% of the training time buys a small fraction
of the performance.